# Bivek Thapa 240495
## PySpark Lab Work: Exploratory Data Analysis (EDA) Using PySpark and Spark SQL
### Class Task

In [1]:
import os
os.environ["PYSPARK_PYTHON"] = "python"
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
from pyspark.sql import functions as F

In [2]:
spark = SparkSession.builder.appName("SuperMarketEDA").master("local[*]").getOrCreate()

## Part A: Data Loading and Inspection
### Q1. Load the supermarket sales dataset into a PySpark DataFrame.


In [3]:
df = spark.read.csv("dataset/supermarket_sales.csv", header=True, inferSchema=True)

### Q2. Display: - First 10 records - Schema of the dataset - Total number of rows and columns

In [4]:
df.limit(10).show()

+--------------+----------+-----------+----------------+------------+--------+----------+-----------+--------------+----------+
|transaction_id|      date|customer_id|product_category|product_name|quantity|unit_price|total_sales|payment_method|      city|
+--------------+----------+-----------+----------------+------------+--------+----------+-----------+--------------+----------+
|         T0001|2025-02-05|       C027|       Beverages|      Coffee|       5|     13.38|       66.9|       eWallet|Biratnagar|
|         T0002|2025-02-25|       C060|       Beverages|  Soft Drink|       1|      2.95|       2.95|       eWallet|Biratnagar|
|         T0003|2025-04-25|       C072|       Beverages|         Tea|       9|     21.85|     196.65|          Cash|   Pokhara|
|         T0004|2025-03-28|       C027|          Snacks|   Chocolate|       5|      9.04|       45.2|          Cash| Bhaktapur|
|         T0005|2025-01-12|       C118|       Beverages|       Juice|       6|     30.78|     184.68|   

In [5]:
df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- total_sales: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)



In [6]:
col_count = len(df.columns)
row_count = df.count()

print("Total number of columns:", col_count)
print("Total number of rows:", row_count)

Total number of columns: 10
Total number of rows: 500


### Q3. Check for: - Null values - Duplicate records

In [7]:
from pyspark.sql.functions import col,sum as spark_sum,isnan,when

In [8]:
null_count = df.select([spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns])
null_count.show()

+--------------+----+-----------+----------------+------------+--------+----------+-----------+--------------+----+
|transaction_id|date|customer_id|product_category|product_name|quantity|unit_price|total_sales|payment_method|city|
+--------------+----+-----------+----------------+------------+--------+----------+-----------+--------------+----+
|             0|   0|          0|               0|           0|       0|         0|          0|             0|   0|
+--------------+----+-----------+----------------+------------+--------+----------+-----------+--------------+----+



In [9]:
duplicate_records = df.groupBy(df.columns).count().filter("count > 1")
duplicate_records.show()

+--------------+----+-----------+----------------+------------+--------+----------+-----------+--------------+----+-----+
|transaction_id|date|customer_id|product_category|product_name|quantity|unit_price|total_sales|payment_method|city|count|
+--------------+----+-----------+----------------+------------+--------+----------+-----------+--------------+----+-----+
+--------------+----+-----------+----------------+------------+--------+----------+-----------+--------------+----+-----+



### Q4. Generate descriptive statistics for all numerical columns.

In [10]:
df.describe().show()

+-------+--------------+-----------+----------------+------------+------------------+------------------+-----------------+--------------+---------+
|summary|transaction_id|customer_id|product_category|product_name|          quantity|        unit_price|      total_sales|payment_method|     city|
+-------+--------------+-----------+----------------+------------+------------------+------------------+-----------------+--------------+---------+
|  count|           500|        500|             500|         500|               500|               500|              500|           500|      500|
|   mean|          NULL|       NULL|            NULL|        NULL|             5.524|25.628980000000013|        145.03284|          NULL|     NULL|
| stddev|          NULL|       NULL|            NULL|        NULL|2.9730627649628962|13.991613046066618|120.2808106872146|          NULL|     NULL|
|    min|         T0001|       C002|          Bakery|       Apple|                 1|              1.73|        

# Part B: EDA Using PySpark DataFrame Methods
### Q5. Calculate the total supermarket revenue.

In [11]:
df.select(F.sum("total_sales").alias("total_revenue")).show()


+-------------+
|total_revenue|
+-------------+
|     72516.42|
+-------------+



### Q6. Find the top 5 highest-selling product categories based on total sales.

In [14]:
df.groupBy("product_category") \
    .agg(F.sum("total_sales").alias("total_sales")) \
    .orderBy(F.desc("total_sales")) \
    .limit(5) \
    .show()

+----------------+------------------+
|product_category|       total_sales|
+----------------+------------------+
|          Snacks|15938.829999999998|
|           Dairy|15883.500000000004|
|          Bakery|14209.069999999994|
|       Beverages|14186.970000000005|
|         Produce|12298.050000000001|
+----------------+------------------+



### Q7. Determine: - Average quantity sold - Average unit price - Average sales amount

In [15]:
df.select(
    F.avg("quantity").alias("avg_quantity_sold"),
    F.avg("unit_price").alias("avg_unit_price"),
    F.avg("total_sales").alias("avg_sales_amount")
).show()

+-----------------+------------------+----------------+
|avg_quantity_sold|    avg_unit_price|avg_sales_amount|
+-----------------+------------------+----------------+
|            5.524|25.628980000000013|       145.03284|
+-----------------+------------------+----------------+



### Q8. Find the city generating the highest revenue.

In [16]:
df.groupBy("city") \
    .agg(F.sum("total_sales").alias("city_revenue")) \
    .orderBy(F.desc("city_revenue")) \
    .limit(1) \
    .show()

+---------+------------------+
|     city|      city_revenue|
+---------+------------------+
|Bhaktapur|16650.069999999996|
+---------+------------------+



### Q9. Identify the payment method used most frequently.

In [18]:
df.groupBy("payment_method").count().orderBy(F.desc("count")).limit(1).show()

+--------------+-----+
|payment_method|count|
+--------------+-----+
|          Card|  177|
+--------------+-----+



### Q10. Calculate total sales for each product category.

In [19]:
df.groupBy("product_category").agg(F.sum("total_sales").alias("total_revenue")).orderBy(F.desc("total_revenue")).show()

+----------------+------------------+
|product_category|     total_revenue|
+----------------+------------------+
|          Snacks|15938.829999999998|
|           Dairy|15883.500000000004|
|          Bakery|14209.069999999994|
|       Beverages|14186.970000000005|
|         Produce|12298.050000000001|
+----------------+------------------+



### Q11. Find the top 10 transactions with the highest sales amount.

In [20]:
df.orderBy(F.desc("total_sales")).limit(10).show()

+--------------+----------+-----------+----------------+------------+--------+----------+-----------+--------------+----------+
|transaction_id|      date|customer_id|product_category|product_name|quantity|unit_price|total_sales|payment_method|      city|
+--------------+----------+-----------+----------------+------------+--------+----------+-----------+--------------+----------+
|         T0015|2025-05-16|       C065|          Snacks|    Crackers|      10|     49.81|      498.1|       eWallet| Kathmandu|
|         T0093|2025-05-15|       C073|          Snacks|   Chocolate|      10|     48.87|      488.7|          Cash|   Pokhara|
|         T0100|2025-01-26|       C054|          Bakery|       Bread|      10|     48.07|      480.7|       eWallet|   Pokhara|
|         T0161|2025-05-24|       C085|           Dairy|        Milk|      10|      47.9|      479.0|       eWallet| Kathmandu|
|         T0098|2025-01-22|       C064|           Dairy|      Cheese|      10|     47.53|      475.3|   

### Q12. Create a new column named sales_level: - High → sales > 100 - Medium → sales between 50 and 100 - Low → sales < 50

Count the number of transactions in each sales level.

In [21]:
df_sales_level = df.withColumn(
    "sales_level",
    F.when(col("total_sales") > 100, "High")
     .when((col("total_sales") >= 50) & (col("total_sales") <= 100), "Medium")
     .otherwise("Low")
)

df_sales_level.groupBy("sales_level").count().show()

+-----------+-----+
|sales_level|count|
+-----------+-----+
|       High|  261|
|        Low|  139|
|     Medium|  100|
+-----------+-----+



# Part C: EDA Using Spark SQL

### Q13. Create a temporary SQL view named `sales_view`

In [23]:
df.createOrReplaceTempView("sales_view")

### Q14. Spark SQL queries

#### a) Total revenue generated by the supermarket.

In [24]:
spark.sql("SELECT SUM(total_sales) AS total_revenue FROM sales_view").show()

+-------------+
|total_revenue|
+-------------+
|     72516.42|
+-------------+



#### b) Revenue generated by each city.

In [25]:
spark.sql(
    "SELECT city, SUM(total_sales) AS revenue FROM sales_view GROUP BY city ORDER BY revenue DESC"
).show()

+----------+------------------+
|      city|           revenue|
+----------+------------------+
| Bhaktapur|16650.069999999996|
|  Lalitpur|16323.509999999998|
|   Pokhara|14540.320000000003|
| Kathmandu|          13034.83|
|Biratnagar|11967.690000000002|
+----------+------------------+



#### c) Revenue generated by each product category.

In [26]:
spark.sql(
    "SELECT product_category, SUM(total_sales) AS revenue FROM sales_view GROUP BY product_category ORDER BY revenue DESC"
).show()

+----------------+------------------+
|product_category|           revenue|
+----------------+------------------+
|          Snacks|15938.829999999998|
|           Dairy|15883.500000000004|
|          Bakery|14209.069999999994|
|       Beverages|14186.970000000005|
|         Produce|12298.050000000001|
+----------------+------------------+



#### d) Average quantity sold for each category.

In [27]:
spark.sql(
    "SELECT product_category, AVG(quantity) AS avg_quantity_sold FROM sales_view GROUP BY product_category ORDER BY avg_quantity_sold DESC"
).show()

+----------------+------------------+
|product_category| avg_quantity_sold|
+----------------+------------------+
|           Dairy| 5.904255319148936|
|          Bakery|5.6020408163265305|
|         Produce| 5.540816326530612|
|          Snacks|             5.375|
|       Beverages| 5.245283018867925|
+----------------+------------------+



#### e) Top 5 products by total sales.

In [28]:
spark.sql(
    "SELECT product_name, SUM(total_sales) AS total_sales FROM sales_view GROUP BY product_name ORDER BY total_sales DESC LIMIT 5"
).show()

+------------+------------------+
|product_name|       total_sales|
+------------+------------------+
|         Tea| 5265.120000000001|
|        Milk| 5056.429999999999|
|     Cookies| 4913.889999999999|
|       Apple|4596.6100000000015|
|      Yogurt| 4299.000000000001|
+------------+------------------+



#### f) Number of transactions for each payment method.

In [29]:
spark.sql(
    "SELECT payment_method, COUNT(*) AS transaction_count FROM sales_view GROUP BY payment_method ORDER BY transaction_count DESC"
).show()

+--------------+-----------------+
|payment_method|transaction_count|
+--------------+-----------------+
|          Card|              177|
|          Cash|              163|
|       eWallet|              160|
+--------------+-----------------+



#### g) Highest single transaction amount.

In [30]:
spark.sql("SELECT MAX(total_sales) AS highest_transaction_amount FROM sales_view").show()

+--------------------------+
|highest_transaction_amount|
+--------------------------+
|                     498.1|
+--------------------------+



#### h) Daily sales summary: - Date - Total Revenue - Number of Transactions

In [31]:
spark.sql(
    "SELECT date, SUM(total_sales) AS total_revenue, COUNT(*) AS transaction_count FROM sales_view GROUP BY date ORDER BY date"
).show()

+----------+------------------+-----------------+
|      date|     total_revenue|transaction_count|
+----------+------------------+-----------------+
|2025-01-01|185.85000000000002|                2|
|2025-01-02|             223.3|                1|
|2025-01-03|            111.04|                1|
|2025-01-04|            688.55|                4|
|2025-01-05|             160.4|                1|
|2025-01-07|            132.75|                1|
|2025-01-08|            433.02|                2|
|2025-01-09|            828.78|                4|
|2025-01-10|392.63000000000005|                3|
|2025-01-11|              43.2|                1|
|2025-01-12|             526.3|                3|
|2025-01-13|            621.75|                3|
|2025-01-14| 897.1800000000001|                4|
|2025-01-15|            945.99|                5|
|2025-01-16|            658.35|                2|
|2025-01-17|            229.68|                2|
|2025-01-18| 852.9499999999999|                5|


# Part D: Insights and Interpretation

- The supermarket’s total revenue reflects the overall business scale and can be used to measure growth over time.
- Cities with the highest revenue are strategic markets where targeted promotions and inventory availability should be prioritized.
- The top product categories carry the largest revenue share, indicating core strengths and the best candidates for upselling.
- Categories with higher average quantities sold show customer demand for larger purchases, which can guide bundle or discount strategies.
- Payment method trends signal customer preferences and can inform checkout process optimization or payment-related promotions.
- The maximum transaction value highlights high-spend customer behavior and suggests opportunities for premium offers.
- Daily sales summaries reveal demand cycles and help optimize staffing and stock levels for peak selling days.